<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# FABRIC Artifact Manager: Package and Share Experiments

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** The FABRIC [Artifact Manager](https://artifacts.fabric-testbed.net) lets you package, share, and reuse complete, repeatable FABRIC experiments. This notebook walks you through the full artifact lifecycle: listing available artifacts, creating a new one, uploading content, and deleting artifacts when they are no longer needed.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. **List** all available artifacts (your own, public, and project-scoped)
2. **Create** a new artifact with metadata (title, description, tags, visibility, authors)
3. **Upload** files (e.g., tar archives) to an artifact
4. **Filter** artifact listings to find specific artifacts
5. **Delete** artifacts when they are no longer needed
6. Understand FABRIC's artifact **visibility levels** (author, project, public)

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Have an active FABRIC project membership

**Tip:** The file `hello_fabric.tgz` referenced in the upload step should exist in this notebook's directory. If it does not, create a sample tar file or substitute your own.

</div>

## Background: FABRIC Artifacts

### What is an Artifact?

A FABRIC **artifact** is a versioned, shareable package that can contain experiment code, configurations, data, and documentation. Artifacts enable:

- **Reproducibility**: Others can re-run your exact experiment
- **Collaboration**: Share experiments with project members or the public
- **Versioning**: Upload updated content as new versions

### Visibility Levels

| Level | Who Can Access | Use Case |
|-------|---------------|----------|
| **author** | Only you (the creator) | Work in progress, private experiments |
| **project** | All members of your FABRIC project | Team collaboration |
| **public** | Anyone with FABRIC access | Published, shareable experiments |

### Artifact Lifecycle



---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library and create a manager instance
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: List Available Artifacts

The `list_artifacts()` method displays all artifacts you have access to, including:
- Artifacts you created
- Public artifacts from other users
- Artifacts shared with your project

In [ ]:
# List all accessible artifacts
fablib.list_artifacts();

## Step 3: Create a New Artifact

To create an artifact, you provide:

- **Title**: A clear, descriptive name for the artifact
- **Short Description**: A brief summary (shown in listings)
- **Long Description**: A detailed explanation of the artifact's purpose and contents
- **Tags**: Keywords for search and categorization
- **Visibility**: Who can access it (`author`, `project`, or `public`)
- **Authors**: List of contributor email addresses (empty list uses your token identity)

In [ ]:
# Define the artifact metadata
artifact_title = "Test-Artifact"
description_short = "Short Description"
description_long = "Long Description"
tags = ["example"]
visibility = "project"  # Options: "author", "project", "public"
authors = []  # Empty list = use your token identity as the author

In [ ]:
# Create the artifact with the specified metadata
artifact = fablib.create_artifact(artifact_title=artifact_title,
                                  description_short=description_short,
                                  description_long=description_long,
                                  tags=tags,
                                  visibility=visibility,
                                  authors=authors)

## Step 4: Verify the Artifact Was Created

We list artifacts again, this time filtering by title to confirm our new artifact exists. The `filter_function` parameter accepts a lambda that filters the artifact list.

In [ ]:
# List only artifacts matching our title
fablib.list_artifacts(filter_function=lambda x: x['title']==artifact_title);

## Step 5: Upload Content to the Artifact

Now we upload a tar archive to the artifact. Each upload creates a new **version** of the artifact, allowing you to track changes over time.

<div class="fab-warning">

**Tip:** Package your experiment as a `.tgz` file containing notebooks, scripts, configuration files, and any small data files needed to reproduce the experiment.

</div>

In [ ]:
# Path to the file to upload
file_to_upload = "./hello_fabric.tgz"

In [ ]:
# Retrieve the artifact object to get its UUID
artifact = fablib.get_artifacts(artifact_title=artifact_title)[0].to_dict()

In [ ]:
# Upload the tar file to the artifact
upload_response = fablib.upload_file_to_artifact(artifact_id=artifact.get("uuid"), 
                                                 file_to_upload=file_to_upload)

print(f"Uploaded tar file to artifact: {upload_response}")

## Step 6: Verify the Upload

After uploading, the artifact listing should now show version information, confirming that content was successfully attached.

In [ ]:
# Verify the artifact now has content (versions should appear)
fablib.list_artifacts(filter_function=lambda x: x['title']==artifact_title);

## Step 7: Delete the Artifact

<div class="fab-danger">

**Warning:** Deleting an artifact is permanent and removes all versions. Make sure you no longer need the artifact before deleting it.

</div>

In [ ]:
# Delete the artifact by title
fablib.delete_artifact(artifact_title=artifact_title);

## Step 8: Verify Deletion

Confirm that the artifact no longer appears in the listing.

In [ ]:
# Verify the artifact has been deleted (should return empty)
fablib.list_artifacts(filter_function=lambda x: x['title']==artifact_title);

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `create_artifact()` fails | Token expired or invalid | Re-run the Configure Environment notebook to refresh your token |
| `list_artifacts()` returns empty | No artifacts exist or token lacks permissions | Verify your project membership and token validity |
| Upload fails with file not found | `hello_fabric.tgz` missing from directory | Create a test tar file: `tar czf hello_fabric.tgz some_file.txt` |
| Cannot see project artifacts | Not a member of the project | Contact your project lead to be added |
| `delete_artifact()` fails | Artifact already deleted or wrong title | Use `list_artifacts()` to verify the exact title |
| Artifact visibility not as expected | Incorrect visibility parameter | Use exactly `"author"`, `"project"`, or `"public"` (lowercase) |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.list_artifacts()` | List all accessible artifacts | [list_artifacts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_artifacts) |
| `fablib.create_artifact(...)` | Create a new artifact with metadata | [create_artifact](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.create_artifact) |
| `fablib.get_artifacts(artifact_title)` | Retrieve artifacts by title | [get_artifacts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_artifacts) |
| `fablib.upload_file_to_artifact(...)` | Upload a file to an artifact | [upload_file_to_artifact](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.upload_file_to_artifact) |
| `fablib.delete_artifact(artifact_title)` | Delete an artifact by title | [delete_artifact](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.delete_artifact) |

## What's Next?

Now that you know how to manage FABRIC artifacts, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Hello, FABRIC** | [hello_fabric](../hello_fabric/hello_fabric.ipynb) | Create your first FABRIC experiment |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **L2 Networking** | [create_l2network_basic_config](../create_l2network_basic/create_l2network_basic_config.ipynb) | Create isolated L2 networks between nodes |
| **Storage Benchmarking** | [benchmarking_storage](../benchmarking_storage/benchmarking_storage.ipynb) | Benchmark local disk and NVMe storage |
| **GPUs** | [fabric_gpu](../fabric_all_gpus/fabric_gpu.ipynb) | Reserve and use NVIDIA GPUs on FABRIC |